# HydroSAR-BD: Complete Replication & Validation Notebook

**Spatiotemporal GMM for Dynamic Surface Water Mapping in Bangladesh (2015–2025)**

## 🔀 Two Modes — Choose One

| Mode | Who is it for? | What it does |
|:---|:---|:---|
| **Mode A (Default — No GEE needed)** | All reviewers | Uses pre-computed data already in the repo. Run All → see all results instantly. |
| **Mode B (Optional — GEE Advanced)** | Reviewers who want to re-derive data from scratch | Authenticates GEE, exports fresh SAR histograms, re-runs full pipeline. |

> **Recommended:** Run Mode A first. It reproduces all figures, tables, and accuracy metrics in < 5 minutes with zero configuration.

---

## Contents
| Section | Description |
|:---|:---|
| **1** | Environment setup (auto-detects Colab) |
| **2** | [Mode A] Load pre-computed data & display all results |
| **3** | ST-GMM threshold calibration |
| **4** | Water area computation |
| **5** | AIC/BIC GMM justification |
| **6** | Per-class accuracy (Permanent / Semi-permanent / Ephemeral) |
| **7** | Publication figures |
| **8** | [Mode B — Optional] GEE live data extraction |


## Section 1 — Environment Setup

In [ ]:
import sys, subprocess, os, ast, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, Image as IPImage

# Auto-install packages
for p in ["pandas","numpy","scikit-learn","scipy","matplotlib"]:
    try: __import__(p.replace("-","_").replace("scikit_learn","sklearn"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",p])

from sklearn.mixture import GaussianMixture
from scipy import stats as scipy_stats
from scipy.stats import norm
warnings.filterwarnings('ignore')

# Auto-detect Google Colab and clone repo
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('SAR'):
        print("Cloning HydroSAR-BD repository...")
        os.system("git clone https://github.com/DruboPaul/SAR.git")
    os.chdir('SAR')
    print("Working directory:", os.getcwd())

BASE_DIR     = os.getcwd()
DATA_DIR     = os.path.join(BASE_DIR, "data")
RESULTS_DIR  = os.path.join(BASE_DIR, "results")
FIGURES_DIR  = os.path.join(BASE_DIR, "figures")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

HIST_CSV  = os.path.join(DATA_DIR, "Bangladesh_District_VV_Histograms_2015_2025.csv")
OCCUR_CSV = os.path.join(DATA_DIR, "Validation_Points_With_Occurrence.csv")
MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
print("✓ Setup complete. Ready to run.")


## Section 2 — Mode A: Pre-Computed Results (Recommended)

> **No GEE account needed.** All outputs were pre-computed using the full pipeline and are stored in `results/`.  
> Run this section to instantly view all manuscript figures and accuracy tables.


In [ ]:
print("=" * 60)
print("  MODE A — LOADING PRE-COMPUTED RESULTS")
print("=" * 60)

# ── Per-class accuracy table ────────────────────────────────────
acc_csv = os.path.join(RESULTS_DIR, "per_class_accuracy.csv")
if os.path.exists(acc_csv):
    acc_df = pd.read_csv(acc_csv)
    print("\n📊 Per-Class Accuracy Assessment (Table from manuscript):")
    print(acc_df.to_string(index=False))
else:
    print("[INFO] per_class_accuracy.csv not found — run Section 6 to generate.")

# ── AIC/BIC scores ──────────────────────────────────────────────
aic_csv = os.path.join(RESULTS_DIR, "GMM_AIC_BIC_Scores.csv")
if os.path.exists(aic_csv):
    aic_df = pd.read_csv(aic_csv)
    print("\n📊 GMM AIC/BIC Information Criteria:")
    print(aic_df.to_string(index=False))
else:
    print("[INFO] GMM_AIC_BIC_Scores.csv not found — run Section 5 to generate.")

# ── Display pre-computed figures ────────────────────────────────
figs = {
    "5-Panel Comparative Map": os.path.join(RESULTS_DIR, "Figure_5Panel_Comparative_Map.png"),
    "GMM AIC/BIC Diagnostic":  os.path.join(RESULTS_DIR, "GMM_AIC_BIC_Test_Plot.png"),
}
for title, fpath in figs.items():
    if os.path.exists(fpath):
        print(f"\n🖼️  {title}:")
        display(IPImage(filename=fpath, width=900))
    else:
        print(f"[INFO] {title} not found at {fpath}")

print("\n✓ Mode A complete. All pre-computed results displayed.")


## Section 3 — ST-GMM Threshold Calibration

Fits a 2-component GMM to each district-month SAR backscatter histogram. The water/land boundary is the intersection point of the two Gaussian components.

In [ ]:
def fit_gmm_threshold(counts, bins):
    mask = counts > 0
    counts, bins = counts[mask], bins[mask]
    if len(bins) < 5 or counts.sum() < 100: return np.nan
    samples = np.repeat(bins, counts.astype(int)).reshape(-1, 1)
    try:
        gmm = GaussianMixture(n_components=2, covariance_type='full', max_iter=200, random_state=42)
        gmm.fit(samples)
        means = gmm.means_.flatten(); stds = np.sqrt(gmm.covariances_.flatten())
        weights = gmm.weights_.flatten(); idx = np.argsort(means)
        means, stds, weights = means[idx], stds[idx], weights[idx]
        x = np.linspace(bins.min(), bins.max(), 1000)
        pdf_w = weights[0] * norm.pdf(x, means[0], stds[0])
        pdf_l = weights[1] * norm.pdf(x, means[1], stds[1])
        ms = (x > means[0]) & (x < means[1])
        if not ms.any(): return float((means[0]*stds[1]+means[1]*stds[0])/(stds[0]+stds[1]))
        diff = pdf_w[ms] - pdf_l[ms]; sc = np.where(np.diff(np.sign(diff)))[0]
        return float(x[ms][sc[0]]) if len(sc) else float((means[0]*stds[1]+means[1]*stds[0])/(stds[0]+stds[1]))
    except: return np.nan

if not os.path.exists(HIST_CSV):
    print(f"[SKIPPED] {HIST_CSV} not found. Run Mode B (Section 8) to export from GEE first.")
else:
    print("Loading 55 MB histogram CSV …")
    df_hist = pd.read_csv(HIST_CSV)
    df_hist['hist'] = df_hist['histogram_counts'].apply(ast.literal_eval)
    df_hist['bins'] = df_hist['histogram_means'].apply(ast.literal_eval) if 'histogram_means' in df_hist.columns                       else [np.linspace(-30, 5, len(h)) for h in df_hist['hist']]
    print(f"  Loaded {len(df_hist):,} district-month rows")
    df_hist['threshold'] = df_hist.apply(
        lambda r: fit_gmm_threshold(np.array(r['hist']), np.array(r['bins'])), axis=1)
    n_fail = df_hist['threshold'].isna().sum()
    print(f"  Converged: {len(df_hist)-n_fail}/{len(df_hist)} | Failed: {n_fail}")
    lookup = df_hist.groupby(['district_name','month'])['threshold'].mean().reset_index()
    lookup.to_csv(os.path.join(RESULTS_DIR,"GMM_Threshold_Lookup.csv"), index=False)
    print("✓ GMM thresholds saved → results/GMM_Threshold_Lookup.csv")
    print(lookup.head(8).to_string(index=False))


## Section 4 — Surface Water Area Computation

Applies GMM thresholds to histograms. Pixel scale = 100 m → 0.01 km² per pixel. Missing entries filled by linear temporal interpolation.

In [ ]:
PIXEL_AREA = (100**2)/1e6

def water_km2(bc, ct, th):
    bc, ct = np.array(bc), np.array(ct); n = min(len(bc),len(ct))
    return float(np.sum(ct[:n][bc[:n] <= th])) * PIXEL_AREA

if os.path.exists(HIST_CSV) and 'lookup' in dir():
    tl = {(r.district_name,r.month): r.threshold for _,r in lookup.iterrows()}
    fb = df_hist.groupby('month')['threshold'].mean().to_dict()
    recs = []
    for _,row in df_hist.iterrows():
        th = tl.get((row['district_name'],row['month']), fb.get(row['month'], -12.0))
        if pd.isna(th): continue
        recs.append({'year':row['year'],'month':row['month'],
                     'district':row['district_name'],
                     'water_area_km2': water_km2(row['bins'],row['hist'],th)})
    nat = pd.DataFrame(recs).groupby(['year','month'])['water_area_km2'].sum().reset_index()
    nat.to_csv(os.path.join(RESULTS_DIR,"national_monthly_water_area.csv"), index=False)
    pk = nat.loc[nat.water_area_km2.idxmax()]
    print(f"✓ Water area saved ({len(nat)} rows)")
    print(f"  Peak: {pk.water_area_km2:,.0f} km² — Year {int(pk.year)}, Month {MONTH_NAMES[int(pk.month)]}")
else:
    print("[SKIPPED] Run Section 3 first, or run Mode B (Section 8) to get data from GEE.")


## Section 5 — GMM Component Justification (AIC / BIC)

Compares 2, 3, 4, 5-component GMMs. Lower score = better fit. The 2-component model is physically interpretable as water vs. non-water.

In [ ]:
if not os.path.exists(HIST_CSV):
    print("[SKIPPED] Histogram CSV not found. Pre-computed AIC/BIC plot shown in Section 2.")
else:
    sample_districts = ['Sunamganj','Dhaka','Bhola']
    aic_results, fig, axes = [], *plt.subplots(1,3,figsize=(16,5)),
    for ax, dist in zip(axes, sample_districts):
        sub = df_hist[df_hist['district_name']==dist]
        row = sub[sub['month']==8].iloc[0] if not sub[sub['month']==8].empty else sub.iloc[0]
        bc = np.array(row['bins']); ct = np.array(row['hist']); mask = ct>0
        sc = max(1, int(ct[mask].sum()//100000))
        s = np.repeat(bc[mask], (ct[mask]/sc).astype(int)).reshape(-1,1)
        aic_s, bic_s = [], []
        for n in [2,3,4,5]:
            g = GaussianMixture(n_components=n,covariance_type='full',max_iter=300,random_state=42).fit(s)
            aic_s.append(g.aic(s)); bic_s.append(g.bic(s))
            aic_results.append({'District':dist,'Components':n,'AIC':g.aic(s),'BIC':g.bic(s)})
        ax.plot([2,3,4,5],aic_s,'o-',lw=2,label='AIC')
        ax.plot([2,3,4,5],bic_s,'s--',lw=2,label='BIC')
        ax.axvline(2,color='red',lw=1.5,ls=':',alpha=0.7,label='Selected (n=2)')
        ax.set_title(dist,fontsize=13,fontweight='bold')
        ax.set_xlabel('GMM Components'); ax.legend(); ax.grid(alpha=0.3,ls='--')
    plt.suptitle('AIC & BIC — GMM Component Selection',fontsize=14,fontweight='bold')
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR,'GMM_AIC_BIC_Test_Plot.png')
    fig.savefig(out,dpi=300); plt.show()
    pd.DataFrame(aic_results).to_csv(os.path.join(RESULTS_DIR,'GMM_AIC_BIC_Scores.csv'),index=False)
    print("✓ AIC/BIC saved → results/")


## Section 6 — Per-Class Accuracy Assessment

Stratifies 4,310 validation points into Permanent (≥80%), Semi-permanent (40–79%), and Ephemeral (1–39%) water classes using JRC occurrence frequency, then computes UA, PA, and OA.

In [ ]:
if not os.path.exists(OCCUR_CSV):
    print(f"[SKIPPED] {OCCUR_CSV} not found.")
else:
    df_val = pd.read_csv(OCCUR_CSV)
    df_val['occurrence'] = df_val['occurrence'].fillna(0)
    def hp(o):
        return 'Permanent' if o>=80 else 'Semi-permanent' if o>=40 else 'Ephemeral' if o>0 else 'Non-water'
    df_val['Water_Class'] = df_val['occurrence'].apply(hp)
    rows = []
    for cls in ['Permanent','Semi-permanent','Ephemeral']:
        s = df_val[df_val['Water_Class']==cls]
        if len(s)==0: continue
        yt, yp = s['Field_Truth'], s['class']
        tp=((yt==1)&(yp==1)).sum(); fp=((yt==0)&(yp==1)).sum()
        fn=((yt==1)&(yp==0)).sum(); tn=((yt==0)&(yp==0)).sum()
        rows.append({'Water Class':cls,'N':len(s),'TP':tp,'FP':fp,'FN':fn,'TN':tn,
                     "UA (%)": round(tp/(tp+fp)*100 if tp+fp else 0,2),
                     "PA (%)": round(tp/(tp+fn)*100 if tp+fn else 0,2),
                     "OA (%)": round((tp+tn)/len(s)*100,2)})
    acc = pd.DataFrame(rows)
    acc.to_csv(os.path.join(RESULTS_DIR,'per_class_accuracy.csv'),index=False)
    print("=" * 60)
    print("  PER-CLASS ACCURACY (manuscript Table)")
    print("=" * 60)
    print(acc.to_string(index=False))
    print("✓ Saved → results/per_class_accuracy.csv")


## Section 7 — Publication Figures

Generates the seasonal ribbon and July peak trend figures from the computed water area time series.

In [ ]:
nat_csv = os.path.join(RESULTS_DIR,'national_monthly_water_area.csv')
if not os.path.exists(nat_csv):
    print("[SKIPPED] Run Section 4 first to generate water area data.")
else:
    nat = pd.read_csv(nat_csv)
    ML = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    stats = nat.groupby('month')['water_area_km2'].agg(['mean','std','min','max']).sort_index()
    x = np.arange(12); mv,sv,nv,xv = stats['mean'].values,stats['std'].values,stats['min'].values,stats['max'].values
    # Seasonal ribbon
    fig,ax = plt.subplots(figsize=(11,5))
    for s,e,c,n in [(0,2,'#E8F4FD','Dry Winter'),(2,5,'#FFF8E1','Pre-Monsoon'),
                    (5,9,'#FFEBEE','Monsoon'),(9,11,'#E8F5E9','Post-Monsoon')]:
        ax.axvspan(s-.5,e-.5,alpha=.12,color=c)
        ax.text((s+e)/2-.5,max(xv)*1.1,n,ha='center',fontsize=8,fontstyle='italic',color='#666')
    ax.fill_between(x,nv,xv,alpha=.10,color='#1f77b4',label='Min-Max (11 yr)')
    ax.fill_between(x,mv-sv,mv+sv,alpha=.22,color='#1f77b4',label='Mean ± 1 SD')
    ax.plot(x,mv,'o-',color='#1f77b4',lw=2.2,ms=7,mfc='white',mew=2,label='11-year Mean')
    ax.set_xticks(x); ax.set_xticklabels(ML)
    ax.set_ylabel('Surface Water Area (km²)'); ax.set_xlabel('Month')
    ax.set_title('Mean Monthly Surface Water Area — Bangladesh (2015–2025)')
    ax.legend(loc='lower left'); ax.grid(True,alpha=.2,ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout(); p1=os.path.join(FIGURES_DIR,'fig_seasonal_ribbon.png')
    fig.savefig(p1,dpi=300); plt.show(); print(f"✓ Saved: {p1}")
    # July trend
    july = nat[nat['month']==7].sort_values('year')
    yrs = july['year'].values.astype(float); area = july['water_area_km2'].values.astype(float)
    sl,ic,r,p,_ = scipy_stats.linregress(yrs,area)
    fig,ax = plt.subplots(figsize=(10,5))
    ax.scatter(yrs,area,color='#1f77b4',s=90,zorder=5,edgecolors='white',lw=1.5)
    ax.plot(yrs,area,'-',color='#1f77b4',alpha=.4,lw=1.5)
    ax.plot(yrs,sl*yrs+ic,'--',color='#d62728',lw=2.5,
            label=f'Trend: {sl:+.1f} km²/yr  (R²={r**2:.3f}, p={p:.3f})')
    ax.set_xlabel('Year'); ax.set_ylabel('Peak Water Area — July (km²)')
    ax.set_title('Decadal Trend in Peak Monsoon Water Extent (July, 2015–2025)')
    ax.legend(); ax.grid(True,alpha=.2,ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout(); p2=os.path.join(FIGURES_DIR,'fig_july_trend.png')
    fig.savefig(p2,dpi=300); plt.show(); print(f"✓ Saved: {p2}")


## Section 8 — Mode B: Live GEE Data Extraction (Optional / Advanced)

> ⚠️ **This section is optional.** It requires a Google Earth Engine account.  
> Skip this section if you are using Mode A (pre-computed data from Section 2).  
> If you want to re-derive the raw SAR histogram data from scratch from GEE, run the cells below.

**Steps:**
1. Authenticate your GEE account (cell below)  
2. The GEE JavaScript export scripts are in `gee_scripts/` — run them in [code.earthengine.google.com](https://code.earthengine.google.com)  
3. Download exported CSVs from your Google Drive into `data/`  
4. Re-run Sections 3–7 to regenerate all results from the fresh data


In [ ]:
# ── OPTIONAL: GEE Authentication ───────────────────────────────
# Uncomment and run this cell ONLY if you want to authenticate GEE.
# Requires earthengine-api: pip install earthengine-api

# import ee
# ee.Authenticate()   # Opens browser for Google sign-in
# ee.Initialize()
# print("✓ GEE authenticated successfully.")
# print()
# print("Next steps:")
# print("  1. Open gee_scripts/01_export_11yr_histograms.js in code.earthengine.google.com")
# print("  2. Click Run → Tasks tab → Run export task")
# print("  3. Download the exported CSV from Google Drive")
# print("  4. Place it in: data/Bangladesh_District_VV_Histograms_2015_2025.csv")
# print("  5. Re-run Sections 3–7 to regenerate all results")
print("Mode B instructions loaded. Uncomment the lines above to authenticate GEE.")
print("GEE scripts available in: gee_scripts/")
import os
gee_dir = os.path.join(os.getcwd(), "gee_scripts")
if os.path.exists(gee_dir):
    for f in sorted(os.listdir(gee_dir)):
        print(f"  → {f}")
